In [ ]:
import pandas as pd
import os

In [ ]:
data_path = '/Users/jk1/stroke_datasets/Extraction_20220815'
pv_prefix = 'patientvalue'
scale_prefix = 'scale'


In [ ]:
patient_value_files = [pd.read_csv(os.path.join(data_path, f), delimiter=';', encoding='utf-8', dtype=str)
             for f in os.listdir(data_path)
             if f.startswith(pv_prefix) and f.endswith('.csv')]

In [ ]:
scale_value_files = [pd.read_csv(os.path.join(data_path, f), delimiter=';', encoding='utf-8', dtype=str)
             for f in os.listdir(data_path)
             if f.startswith(scale_prefix) and f.endswith('.csv')]

identification of duration of continuous monitoring 
- continuous monitoring is defined as a pulse measured at least every hour (pv.pulse) 

In [ ]:
# Compute continuous pulse-monitoring duration per patient.
# Continuous monitoring: no gap > 1 hour between consecutive pulse measurements.
pulse_frames = []
for df in patient_value_files:
    mask = (df['patient_value'] == 'pv.pulse') & df['patient_id'].notna() & df['datetime'].notna()
    if mask.any():
        pulse_frames.append(df.loc[mask, ['patient_id', 'datetime']].copy())

if not pulse_frames:
    duration_by_patient = pd.DataFrame(
        columns=[
            'patient_id',
            'n_pulse_measurements',
            'n_monitoring_segments',
            'longest_continuous_monitoring_hours',
            'total_continuous_monitoring_hours',
            'first_pulse_time',
            'last_pulse_time'
        ]
    )
else:
    pulse_df = pd.concat(pulse_frames, ignore_index=True)
    pulse_df['datetime'] = pd.to_datetime(pulse_df['datetime'], dayfirst=True, errors='coerce')
    pulse_df = pulse_df.dropna(subset=['datetime']).drop_duplicates()
    pulse_df = pulse_df.sort_values(['patient_id', 'datetime'])

    one_hour = pd.Timedelta(hours=1)

    def summarize_patient(group):
        times = group['datetime'].sort_values().drop_duplicates().reset_index(drop=True)
        gaps = times.diff()
        segment_id = gaps.gt(one_hour).cumsum()
        segments = pd.DataFrame({'datetime': times, 'segment_id': segment_id})

        seg_summary = segments.groupby('segment_id')['datetime'].agg(['min', 'max', 'count'])
        seg_durations = (seg_summary['max'] - seg_summary['min']).dt.total_seconds() / 3600

        return pd.Series({
            'n_pulse_measurements': int(len(times)),
            'n_monitoring_segments': int(len(seg_summary)),
            'longest_continuous_monitoring_hours': float(seg_durations.max()) if len(seg_durations) else 0.0,
            'total_continuous_monitoring_hours': float(seg_durations.sum()) if len(seg_durations) else 0.0,
            'first_pulse_time': times.min(),
            'last_pulse_time': times.max()
        })

    duration_by_patient = (
        pulse_df.groupby('patient_id', as_index=False)
        .apply(summarize_patient)
        .reset_index(drop=True)
        .sort_values('longest_continuous_monitoring_hours', ascending=False)
    )

display(duration_by_patient.head(20))
print(f'Patients with pulse data: {len(duration_by_patient)}')

In [ ]:
# mean (SD) longest continuous monitoring hours
print(f'Mean (SD) longest continuous monitoring hours: {duration_by_patient["longest_continuous_monitoring_hours"].mean():.2f} ({duration_by_patient["longest_continuous_monitoring_hours"].std():.2f})')
# median (IQR) longest continuous monitoring hours
median = duration_by_patient['longest_continuous_monitoring_hours'].median()
q1 = duration_by_patient['longest_continuous_monitoring_hours'].quantile(0.25)
q3 = duration_by_patient['longest_continuous_monitoring_hours'].quantile(0.75)
print(f'Median (IQR) longest continuous monitoring hours: {median:.2f} ({q1:.2f} - {q3:.2f})')   

## interval of monitoring of NIHSS scale
- overall
- during continuous monitoring perdio

In [ ]:
NIHSS_scale_name = 'NIHSS - National Institute oh Health Stroke Scale'

In [ ]:
# NIHSS interval analysis: overall hospitalization vs longest continuous monitoring period
scale_frames = []
for df in scale_value_files:
    mask = df['patient_id'].notna() & df['event_date'].notna()
    if mask.any():
        scale_frames.append(df.loc[mask, ['patient_id', 'scale', 'event_date']].copy())

if not scale_frames:
    print('No scale data available.')
else:
    scale_df = pd.concat(scale_frames, ignore_index=True)
    scale_df['event_date'] = pd.to_datetime(scale_df['event_date'], dayfirst=True, errors='coerce')
    scale_df = scale_df.dropna(subset=['event_date'])

    # Robust NIHSS filter in case naming differs slightly across files.
    nihss_mask = scale_df['scale'].astype(str).str.contains('NIHSS', case=False, na=False)
    nihss_df = scale_df.loc[nihss_mask, ['patient_id', 'event_date']].drop_duplicates()
    nihss_df = nihss_df.sort_values(['patient_id', 'event_date'])

    def get_intervals_hours(events_df):
        events_df = events_df.sort_values(['patient_id', 'event_date']).copy()
        events_df['interval_hours'] = (
            events_df.groupby('patient_id')['event_date']
            .diff()
            .dt.total_seconds() / 3600
        )
        return events_df['interval_hours'].dropna()

    def describe_intervals(intervals, label):
        if intervals.empty:
            print(f'{label}: no consecutive NIHSS evaluations found.')
            return
        mean_val = intervals.mean()
        sd_val = intervals.std()
        med_val = intervals.median()
        q1 = intervals.quantile(0.25)
        q3 = intervals.quantile(0.75)
        print(f'{label} - Mean (SD) interval in hours: {mean_val:.2f} ({sd_val:.2f})')
        print(f'{label} - Median (IQR) interval in hours: {med_val:.2f} ({q1:.2f} - {q3:.2f})')
        print(f'{label} - Number of intervals: {len(intervals)}')

    # 1) Overall period (all NIHSS events)
    overall_intervals = get_intervals_hours(nihss_df)
    describe_intervals(overall_intervals, 'Overall period')

    # 2) Continuous monitoring period = longest pulse-continuous segment per patient
    pulse_segments_frames = []
    for df in patient_value_files:
        pulse_mask = (df['patient_value'] == 'pv.pulse') & df['patient_id'].notna() & df['datetime'].notna()
        if pulse_mask.any():
            pulse_segments_frames.append(df.loc[pulse_mask, ['patient_id', 'datetime']].copy())

    if not pulse_segments_frames:
        print('Continuous monitoring period: no pulse data available.')
    else:
        pulse_all = pd.concat(pulse_segments_frames, ignore_index=True)
        pulse_all['datetime'] = pd.to_datetime(pulse_all['datetime'], dayfirst=True, errors='coerce')
        pulse_all = pulse_all.dropna(subset=['datetime']).drop_duplicates()
        pulse_all = pulse_all.sort_values(['patient_id', 'datetime'])

        one_hour = pd.Timedelta(hours=1)
        longest_segments = []
        for pid, g in pulse_all.groupby('patient_id'):
            times = g['datetime'].sort_values().drop_duplicates().reset_index(drop=True)
            gaps = times.diff()
            seg_id = gaps.gt(one_hour).cumsum()
            segs = pd.DataFrame({'datetime': times, 'seg_id': seg_id})
            seg_summary = segs.groupby('seg_id')['datetime'].agg(['min', 'max'])
            seg_summary['duration_h'] = (seg_summary['max'] - seg_summary['min']).dt.total_seconds() / 3600
            best = seg_summary.sort_values('duration_h', ascending=False).iloc[0]
            longest_segments.append({
                'patient_id': pid,
                'cont_start': best['min'],
                'cont_end': best['max'],
                'cont_duration_h': best['duration_h']
            })

        longest_cont_df = pd.DataFrame(longest_segments)

        nihss_in_cont = nihss_df.merge(longest_cont_df[['patient_id', 'cont_start', 'cont_end']], on='patient_id', how='inner')
        in_window_mask = (nihss_in_cont['event_date'] >= nihss_in_cont['cont_start']) & (nihss_in_cont['event_date'] <= nihss_in_cont['cont_end'])
        nihss_in_cont = nihss_in_cont.loc[in_window_mask, ['patient_id', 'event_date']].drop_duplicates()

        cont_intervals = get_intervals_hours(nihss_in_cont)
        describe_intervals(cont_intervals, 'Continuous monitoring period')